In [1]:
import pandas as pd
import numpy as np

In [2]:
df_ote_prices = pd.read_csv("../data/ote_prices_15min.csv", sep=";")
df_czechia_load_prediction = pd.read_csv("../data/czechia_load_prediction_15min.csv", sep=";")
df_germany_load_prediction = pd.read_csv("../data/germany_load_day-ahead_prediction_15min.csv", sep=";")
df_wind_solar_loads_prediction = pd.read_csv("../data/wind_solar_generation_loads_day-ahead_prediction.csv", sep=";")
df_electricity_price_loads = pd.read_csv("../data/electricity_price_loads.csv", sep=";")
df_ote_gas_prices = pd.read_csv("../data/ote_gas_prices.csv", sep=";")
df_eua_prices = pd.read_csv("../data/eua_prices_complete.csv", sep=";")
df_wind_solar_loads_last_week = pd.read_csv("../data/wind_solar_generation_loads_last_week.csv", sep=";")
df_wind_solar_generation_last_week = pd.read_csv("../data/wind_solar_generation_last_week_15min.csv", sep=";")
df_germany_load_last_week = pd.read_csv("../data/germany_load_last_week_15min.csv", sep=";")
df_czechia_load_last_week = pd.read_csv("../data/czechia_load_last_week_15min.csv", sep=";")
df_germany_other_sources_generation_last = pd.read_csv("../data/germany_other_sources_generation_last_week_15min.csv", sep=";")
df_czechia_other_sources_generation_last = pd.read_csv("../data/czechia_other_sources_generation_last_week_15min.csv", sep=";")

In [3]:
df_dataset = pd.DataFrame()

In [4]:
df_dataset["date"] = df_ote_prices[672:]["date"]
df_dataset["period"] = df_ote_prices[672:]["period"]
df_dataset["price"] = df_ote_prices[672:]["price"]
df_dataset["is_peak"] = df_ote_prices[672:]["is_peak"].astype(int)

In [5]:
df_dataset = df_dataset.reset_index(drop=True)

In [6]:
df_dataset.head()

,date,period,price,is_peak
0,2023-01-01,1,4.84,0
1,2023-01-01,2,4.84,0
2,2023-01-01,3,4.84,0
3,2023-01-01,4,4.84,0
4,2023-01-01,5,-0.35,0


In [7]:
len(df_dataset)

105216

In [8]:
df_dataset['intervals_in_day'] = df_dataset.groupby(df_dataset['date'])['period'].transform('max')

df_dataset['sin_time'] = np.sin(2 * np.pi * df_dataset['period'] / df_dataset['intervals_in_day'])
df_dataset['cos_time'] = np.cos(2 * np.pi * df_dataset['period'] / df_dataset['intervals_in_day'])

df_dataset.drop(columns=['intervals_in_day'], inplace=True)

In [9]:
df_dataset["czechia_load_prediction"] = df_czechia_load_prediction["load_prediction"]

In [10]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0


In [11]:
df_dataset["germany_load_prediction"] = df_germany_load_prediction["total_load"]

In [12]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0


In [13]:
df_dataset = pd.merge(df_dataset, df_wind_solar_loads_prediction, on='date', how='left')

In [14]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,solar_offpeak,wind_baseload,wind_peakload,wind_offpeak
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98


In [15]:
df_dataset = pd.merge(df_dataset, df_electricity_price_loads, on='date', how='left')
df_dataset = df_dataset.rename(columns={
    'Baseload Price': 'price_baseload',
    'Peakload Price': 'price_peakload',
    'Off-peak Price': 'price_offpeak',
})

In [16]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,solar_offpeak,wind_baseload,wind_peakload,wind_offpeak,price_baseload,price_peakload,price_offpeak
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95


In [17]:
df_dataset = pd.merge(df_dataset, df_ote_gas_prices, on='date', how='left')

In [18]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,solar_offpeak,wind_baseload,wind_peakload,wind_offpeak,price_baseload,price_peakload,price_offpeak,gas_price
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24


In [19]:
df_dataset = pd.merge(df_dataset, df_eua_prices, on='date', how='left')

In [20]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,solar_offpeak,wind_baseload,wind_peakload,wind_offpeak,price_baseload,price_peakload,price_offpeak,gas_price,eua_price
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97


In [21]:
df_dataset["date"] = pd.to_datetime(df_dataset['date'])
df_dataset["last_week_date"] = df_dataset["date"] - pd.Timedelta(days=7)
df_dataset["date"] = df_dataset['date'].astype(str)
df_dataset["last_week_date"] = df_dataset["last_week_date"].astype(str)

In [22]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,solar_offpeak,wind_baseload,wind_peakload,wind_offpeak,price_baseload,price_peakload,price_offpeak,gas_price,eua_price,last_week_date
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,0.0,29955.36,27228.75,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25


In [23]:
df_dataset = pd.merge(df_dataset, df_electricity_price_loads, left_on='last_week_date', right_on="date", how='left').drop(columns="date_y")
df_dataset = df_dataset.rename(columns={
    'date_x': 'date',
    'Baseload Price': 'lw_price_baseload',
    'Peakload Price': 'lw_price_peakload',
    'Off-peak Price': 'lw_price_offpeak',
})

In [24]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,wind_offpeak,price_baseload,price_peakload,price_offpeak,gas_price,eua_price,last_week_date,lw_price_baseload,lw_price_peakload,lw_price_offpeak
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25,101.42,111.75,91.09
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25,101.42,111.75,91.09
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25,101.42,111.75,91.09
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25,101.42,111.75,91.09
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,32681.98,16.38,18.81,13.95,65.24,83.97,2022-12-25,101.42,111.75,91.09


In [25]:
df_dataset = pd.merge(df_dataset, df_wind_solar_loads_last_week, left_on='last_week_date', right_on="date", how='left').drop(columns="date_y")
df_dataset = df_dataset.rename(columns={
    'date_x': 'date',
    'solar_baseload_last_week': 'lw_solar_baseload',
    'solar_peakload_last_week': 'lw_solar_peakload',
    'solar_offpeak_last_week': 'lw_solar_offpeak',
    'wind_baseload_last_week': 'lw_wind_baseload',
    'wind_peakload_last_week': 'lw_wind_peakload',
    'wind_offpeak_last_week': 'lw_wind_offpeak',
})

In [26]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,last_week_date,lw_price_baseload,lw_price_peakload,lw_price_offpeak,lw_solar_baseload,lw_solar_peakload,lw_solar_offpeak,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,2022-12-25,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,2022-12-25,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,2022-12-25,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,2022-12-25,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,2022-12-25,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06


In [27]:
df_dataset = pd.merge(df_dataset, df_ote_gas_prices, left_on='last_week_date', right_on="date", how='left').drop(columns="date_y")
df_dataset = df_dataset.rename(columns={
    'date_x': 'date',
    'gas_price_x': 'gas_price',
    'gas_price_y': 'lw_gas_price',
})

In [28]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_price_baseload,lw_price_peakload,lw_price_offpeak,lw_solar_baseload,lw_solar_peakload,lw_solar_offpeak,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak,lw_gas_price
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,101.42,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44


In [29]:
df_dataset = pd.merge(df_dataset, df_eua_prices, left_on='last_week_date', right_on="date", how='left').drop(columns="date_y")
df_dataset = df_dataset.rename(columns={
    'date_x': 'date',
    'eua_price_x': 'eua_price',
    'eua_price_y': 'lw_eua_price',
})

In [30]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_price_peakload,lw_price_offpeak,lw_solar_baseload,lw_solar_peakload,lw_solar_offpeak,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak,lw_gas_price,lw_eua_price
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,111.75,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37


In [31]:
df_dataset.drop(columns=['last_week_date'], inplace=True)

In [32]:
def join_last_week(df_main, df_history):
    main = df_main.copy()
    hist = df_history.copy()
    
    main['date'] = pd.to_datetime(main['date'])
    
    start_time = main['date'].min() - pd.Timedelta(weeks=1)
    
    hist_index = pd.date_range(
        start=start_time, 
        periods=len(hist), 
        freq='15min', 
        tz='Europe/Prague'
    )
    
    hist['_h_date'] = hist_index.date.astype('datetime64[ns]')
    hist['_h_period'] = hist.groupby('_h_date').cumcount() + 1
    
    main['_len_m'] = main.groupby('date')['period'].transform('max')
    
    hist_len_map = hist.groupby('_h_date')['_h_period'].max()
    main['_lookup_date'] = main['date'] - pd.Timedelta(weeks=1)
    main['_len_h'] = main['_lookup_date'].map(hist_len_map)

    main['_target_period'] = main['period']
    
    mask_long = (main['_len_m'] == 100) & (main['_len_h'] == 96)
    main.loc[mask_long & (main['period'] > 12), '_target_period'] -= 4
    
    mask_short = (main['_len_m'] == 92) & (main['_len_h'] == 96)
    main.loc[mask_short & (main['period'] > 8), '_target_period'] += 4
    
    mask_hist_long = (main['_len_m'] == 96) & (main['_len_h'] == 100)
    main.loc[mask_hist_long & (main['period'] > 12), '_target_period'] += 4

    mask_hist_short = (main['_len_m'] == 96) & (main['_len_h'] == 92)
    main.loc[mask_hist_short & (main['period'] > 12), '_target_period'] -= 4
    
    main.loc[mask_hist_short & (main['period'].between(9, 12)), '_target_period'] = -1

    merged = main.merge(
        hist,
        left_on=['_lookup_date', '_target_period'],
        right_on=['_h_date', '_h_period'],
        how='left',
        suffixes=('', '_prev')
    )

    cols_to_drop = ['_h_date', '_h_period', '_len_m', '_lookup_date', 
                    '_len_h', '_target_period']
    return merged.drop(columns=cols_to_drop, errors='ignore')

In [33]:
df_dataset = join_last_week(df_dataset, df_ote_prices[:105217])
df_dataset = df_dataset.drop(columns=['date_prev', 'period_prev', 'time_interval', 'is_peak_prev'])
df_dataset = df_dataset.rename(columns={
    'price_prev': 'lw_price',

})

In [34]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_price_offpeak,lw_solar_baseload,lw_solar_peakload,lw_solar_offpeak,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak,lw_gas_price,lw_eua_price,lw_price
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,91.09,6100.17,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,112.11


In [35]:
df_dataset = join_last_week(df_dataset, df_wind_solar_generation_last_week)
df_dataset = df_dataset.rename(columns={
    'germany_solar_gen_last_week': 'lw_germany_solar_gen',
    'germany_wind_gen_last_week': 'lw_germany_wind_gen',
})

In [36]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_solar_peakload,lw_solar_offpeak,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak,lw_gas_price,lw_eua_price,lw_price,lw_germany_solar_gen,lw_germany_wind_gen
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13326.0
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13053.0
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13149.0
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13271.0
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,12181.5,18.83,15741.57,15930.08,15553.06,75.44,89.37,112.11,5.0,13128.0


In [37]:
df_dataset = join_last_week(df_dataset, df_germany_load_last_week)
df_dataset = df_dataset.drop(columns=['start_date', 'end_date'])
df_dataset = df_dataset.rename(columns={
    'total_load': 'lw_germany_load',

})

In [38]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_solar_offpeak,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak,lw_gas_price,lw_eua_price,lw_price,lw_germany_solar_gen,lw_germany_wind_gen,lw_germany_load
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13326.0,38537.0
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13053.0,37901.0
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13149.0,37306.0
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,18.83,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13271.0,36611.0
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,18.83,15741.57,15930.08,15553.06,75.44,89.37,112.11,5.0,13128.0,36150.0


In [39]:
df_dataset = join_last_week(df_dataset, df_czechia_load_last_week)
df_dataset = df_dataset.rename(columns={
    'last_week_load': 'lw_czechia_load',

})

In [40]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_wind_baseload,lw_wind_peakload,lw_wind_offpeak,lw_gas_price,lw_eua_price,lw_price,lw_germany_solar_gen,lw_germany_wind_gen,lw_germany_load,lw_czechia_load
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13326.0,38537.0,5180.49
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13053.0,37901.0,5180.49
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13149.0,37306.0,5180.49
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,15741.57,15930.08,15553.06,75.44,89.37,120.28,5.0,13271.0,36611.0,5180.49
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,15741.57,15930.08,15553.06,75.44,89.37,112.11,5.0,13128.0,36150.0,5094.07


In [41]:
df_dataset = join_last_week(df_dataset, df_germany_other_sources_generation_last)

In [42]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,lw_germany_load,lw_czechia_load,germany_biomass_gen_last_week,germany_hydro_passive_gen_last_week,germany_nuclear_gen_last_week,germany_lignite_gen_last_week,germany_hard_coal_gen_last_week,germany_fossil_gas_gen_last_week,germany_hydro_active_gen_last_week,germany_other_gen_last_week
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,38537.0,5180.49,4326.0,1702.0,3578.0,11483.0,2466.0,2821.0,132.0,1325.0
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,37901.0,5180.49,4336.0,1631.0,3577.0,11106.0,2441.0,2779.0,175.0,1325.0
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,37306.0,5180.49,4339.0,1587.0,3575.0,11001.0,2424.0,2779.0,149.0,1328.0
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,36611.0,5180.49,4341.0,1584.0,3575.0,10833.0,2400.0,2721.0,150.0,1329.0
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,36150.0,5094.07,4316.0,1580.0,3581.0,10565.0,2380.0,2656.0,9.0,1326.0


In [43]:
df_dataset = join_last_week(df_dataset, df_czechia_other_sources_generation_last)

In [44]:
df_dataset.head()

,date,period,price,is_peak,sin_time,cos_time,czechia_load_prediction,germany_load_prediction,solar_baseload,solar_peakload,...,germany_other_gen_last_week,czechia_biomass_gen_last_week,czechia_lignite_gen_last_week,czechia_fossil_gas_gen_last_week,czechia_hard_coal_gen_last_week,czechia_hydro_active_gen_last_week,czechia_hydro_passive_gen_last_week,czechia_hydro_reservoir_gen_last_week,czechia_nuclear_gen_last_week,czechia_other_gen_last_week
0,2023-01-01,1,4.84,0,0.065403,0.997859,5226.0,43046.0,8713.29,17426.58,...,1325.0,282.15,3860.60,349.89,368.05,41.68,95.19,88.54,3927.25,5.04
1,2023-01-01,2,4.84,0,0.130526,0.991445,5226.0,42577.0,8713.29,17426.58,...,1325.0,282.15,3860.60,349.89,368.05,41.68,95.19,88.54,3927.25,5.04
2,2023-01-01,3,4.84,0,0.195090,0.980785,5226.0,41937.0,8713.29,17426.58,...,1328.0,282.15,3860.60,349.89,368.05,41.68,95.19,88.54,3927.25,5.04
3,2023-01-01,4,4.84,0,0.258819,0.965926,5226.0,41302.0,8713.29,17426.58,...,1329.0,282.15,3860.60,349.89,368.05,41.68,95.19,88.54,3927.25,5.04
4,2023-01-01,5,-0.35,0,0.321439,0.946930,5110.0,40841.0,8713.29,17426.58,...,1326.0,278.82,3686.78,349.84,364.29,5.16,96.15,39.74,3933.62,5.41


In [45]:
df_dataset.drop(columns=['date'], inplace=True)

In [46]:
df_dataset.to_csv("../dataset/2023-2025_dataset.csv", sep=";", index=False)